# Import

In [ ]:
!pip install torch_geometric

In [2]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

import pickle

import torch
from torch_geometric.data import Data

import gc
from collections import defaultdict

# Change directory

In [3]:
# Cambiar al directorio
os.chdir('/content/drive/MyDrive/nids-mitre/')

# Verificar que estás ahí
!pwd


/content/drive/MyDrive/nids-mitre


# Explore data

In [ ]:
!unzip /content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964.zip \
  "*/data/NF-CICIDS2018-v3.csv" \
  -d /content/drive/MyDrive/nids-mitre/data/cicids2018-v3


In [ ]:
!head /content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964/data/NF-CICIDS2018-v3.csv

In [ ]:
!awk -F',' 'NR>1 {print $3; print $5}' \
  /content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964/data/NF-CICIDS2018-v3.csv \
  | sort -u | wc -l

205801


In [4]:
# ============================================================================
# ANÁLISIS EXPLORATORIO SIN CARGAR TODO EN MEMORIA
# ============================================================================

# 1. Leer SOLO las primeras filas para ver estructura
df_sample = pd.read_csv('/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964/data/NF-CICIDS2018-v3.csv', nrows=10000)

print("Columnas disponibles:")
print(df_sample.columns.tolist())
print(f"\nShape muestra: {df_sample.shape}")
print(f"\nTipos de datos:")
print(df_sample.dtypes)

# 2. Contar filas TOTALES sin cargar todo
total_rows = sum(1 for _ in open('/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964/data/NF-CICIDS2018-v3.csv')) - 1  # -1 por header
print(f"\n✓ Total de flows: {total_rows:,}")

# 3. Identificar columnas clave
timestamp_col_start = 'FLOW_START_MILLISECONDS'
timestamp_col_end = 'FLOW_END_MILLISECONDS'
src_ip_col = 'IPV4_SRC_ADDR'
dst_ip_col = 'IPV4_DST_ADDR'
label_col = 'Label'

print(f"\nColumnas identificadas:")
print(f"  Timestamp: {timestamp_col_start}")
print(f"  Timestamp: {timestamp_col_end}")
print(f"  Source IP: {src_ip_col}")
print(f"  Dest IP: {dst_ip_col}")
print(f"  Label: {label_col}")

# 4. Ver distribución de labels (sin cargar todo)
print("\nDistribución de labels (muestra):")
print(df_sample[label_col].value_counts())

# 5. Ver rango temporal
df_sample[timestamp_col_start] = pd.to_datetime(df_sample[timestamp_col_start])
print(f"\nRango temporal (muestra):")
print(f"  Inicio: {df_sample[timestamp_col_start].min()}")
print(f"  Fin: {df_sample[timestamp_col_start].max()}")

Columnas disponibles:
['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR', 'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE', 'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE', 'SRC_TO_DST_IAT_MIN', 'SRC_TO_DST_IAT_MAX', 'SRC_TO_DST_IAT_AVG', 'SR

# Filter and save chunks

In [5]:
selected_ips_list = pd.read_csv('/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/ip_statistics.csv')
# NOTE: I used all ips, not only the most relevant (!!)

In [ ]:
# ============================================================================
# GUARDAR CHUNKS FILTRADOS INCREMENTALMENTE (RECOMENDADA)
# ============================================================================

def filter_and_save_incrementally(csv_path, selected_ips, timestamp_col,
                                  src_ip_col, dst_ip_col, label_col,
                                  output_file,
                                  chunk_size):
    """
    Filtra y guarda datos incrementalmente SIN concatenar en memoria
    """

    total_rows_kept = 0
    chunk_count = 0

    print("Filtrando y guardando por chunks...")
    print(f"Archivo de salida: {output_file}")

    # Abrir archivo de salida en modo append binario
    with open(output_file, 'wb') as f_out:

        for i, chunk in enumerate(pd.read_csv(csv_path, chunksize=chunk_size)):
            print(i, len(chunk))
            print(f"  Procesando chunk {i+1}... ({total_rows_kept:,} rows guardadas)", end='\r')

            # Filtrar
            mask = (
                chunk[src_ip_col].isin(selected_ips['ip']) |
                chunk[dst_ip_col].isin(selected_ips['ip'])
            )

            filtered = chunk[mask].copy()

            if len(filtered) > 0:
                # Convertir timestamp
                filtered[timestamp_col] = pd.to_datetime(filtered[timestamp_col])

                # Guardar este chunk
                pickle.dump(filtered, f_out)

                total_rows_kept += len(filtered)
                chunk_count += 1

            # Liberar memoria
            del chunk, filtered, mask

    print(f"\n✓ Filtrado completado:")
    print(f"  Total rows: {total_rows_kept:,}")
    print(f"  Chunks guardados: {chunk_count}")
    print(f"  Tamaño archivo: {os.path.getsize(output_file) / (1024**2):.2f} MB")

    return total_rows_kept, chunk_count

# Ejecutar filtrado
total_rows, num_chunks = filter_and_save_incrementally(
    '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964/data/NF-CICIDS2018-v3.csv',
    selected_ips_list,
    timestamp_col_start,
    src_ip_col,
    dst_ip_col,
    label_col,
    output_file='/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/filtered_chunks.pkl',
    chunk_size=250000
)

Filtrando y guardando por chunks...
Archivo de salida: /content/drive/MyDrive/nids-mitre/data/cicids2018-v3/filtered_chunks.pkl
0 250000
1 250000
2 250000
3 250000
4 250000
5 250000
6 250000
7 250000
8 250000
9 250000
10 250000
11 250000
12 250000
13 250000
14 250000
15 250000
16 250000
17 250000
18 250000
19 250000
20 250000
21 250000
22 250000
23 250000
24 250000
25 250000
26 250000
27 250000
28 250000
29 250000
30 250000
31 250000
32 250000
33 250000
34 250000
35 250000
36 250000
37 250000
38 250000
39 250000
40 250000
41 250000
42 250000
43 250000
44 250000
45 250000
46 250000
47 250000
48 250000
49 250000
50 250000
51 250000
52 250000
53 250000
54 250000
55 250000
56 250000
57 250000
58 250000
59 250000
60 250000
61 250000
62 250000
63 250000
64 250000
65 250000
66 250000
67 250000
68 250000
69 250000
70 250000
71 250000
72 250000
73 250000
74 250000
75 250000
76 250000
77 250000
78 250000
79 250000
80 115529

✓ Filtrado completado:
  Total rows: 20,115,529
  Chunks guardados: 81


# Read chunks

In [6]:
# ============================================================================
# LEER CHUNKS GUARDADOS UNO A LA VEZ
# ============================================================================

def read_filtered_chunks(pickle_file):
    """
    Generador que lee chunks de uno en uno del archivo pickle
    """

    with open(pickle_file, 'rb') as f:
        while True:
            try:
                chunk = pickle.load(f)
                yield chunk
            except EOFError:
                break

# Ejemplo de uso:
print("\nProbando lectura de chunks...")
for i, chunk in enumerate(read_filtered_chunks('/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/filtered_chunks.pkl')):
    print(f"Chunk {i+1}: {chunk.shape}")
    if i >= 2:  # Solo mostrar primeros 3 chunks
        print("...")
        break


Probando lectura de chunks...
Chunk 1: (250000, 55)
Chunk 2: (250000, 55)
Chunk 3: (250000, 55)
...


# Create graphs

In [ ]:
# ============================================================================
# IMPLEMENTACIÓN COMPLETA: GRAFOS CON NODOS FIJOS PARA GNN+LSTM
# ============================================================================



# ============================================================================
# FUNCIÓN 1: Crear grafos con nodos fijos
# ============================================================================

def create_graphs_with_fixed_nodes(pickle_file, timestamp_col, src_ip_col,
                                   dst_ip_col, label_col,
                                   window_minutes=10,
                                   output_dir='graph_data'):
    """
    Crea grafos temporales con conjunto FIJO de nodos para GNN+LSTM

    Pipeline:
    1. Identificar todas las IPs únicas (universo de nodos)
    2. Crear mapeo global IP → índice de nodo
    3. Para cada ventana temporal:
       - Todos los grafos tienen TODOS los nodos
       - Solo las aristas cambian según flows presentes
       - Nodos sin actividad tienen features = 0

    Returns:
        num_graphs: Número de grafos creados
        metadata: Diccionario con info del dataset
    """

    print("="*70)
    print(" 🔷 CREACIÓN DE GRAFOS CON NODOS FIJOS PARA GNN+LSTM")
    print("="*70)

    # Crear directorio de salida
    os.makedirs(output_dir, exist_ok=True)

    # ========================================================================
    # PASADA 0: Identificar UNIVERSO de IPs (nodos fijos)
    # ========================================================================

    print("\n📊 PASADA 0: Identificando universo de nodos...")
    print("  (Esto determina el número FIJO de nodos en todos los grafos)\n")

    all_ips = set()
    chunk_count = 0

    for chunk in read_filtered_chunks(pickle_file):
        chunk_count += 1
        print(f"  Chunk {chunk_count}... (IPs acumuladas: {len(all_ips):,})")

        # Corregir timestamps
        ns_vals = chunk[timestamp_col].values.astype('int64')
        chunk[timestamp_col] = pd.to_datetime(ns_vals * 1e6, unit='ns')

        # Recolectar IPs
        all_ips.update(chunk[src_ip_col].unique())
        all_ips.update(chunk[dst_ip_col].unique())

        del chunk

    all_ips = sorted(list(all_ips))  # Ordenar para consistencia
    num_nodes = len(all_ips)

    # Crear mapeo GLOBAL (fijo para TODOS los grafos)
    global_ip_to_idx = {ip: idx for idx, ip in enumerate(all_ips)}

    print(f"\n\n✅ Universo de nodos identificado:")
    print(f"   Total IPs únicas: {num_nodes:,}")
    print(f"   Cada grafo tendrá exactamente {num_nodes:,} nodos")

    # Guardar mapeo
    mapping_file = f'{output_dir}/ip_to_idx_mapping.pkl'
    with open(mapping_file, 'wb') as f:
        pickle.dump(global_ip_to_idx, f)
    print(f"   Mapeo guardado en: {mapping_file}")

    # ========================================================================
    # PASADA 1: Identificar rango temporal y ventanas
    # ========================================================================

    print("\n📊 PASADA 1: Identificando rango temporal...")

    start_time = None
    end_time = None
    total_chunks = 0

    for chunk in read_filtered_chunks(pickle_file):
        total_chunks += 1
        print(f"  Chunk {total_chunks}...", end='\r')

        # Corregir timestamps
        ns_vals = chunk[timestamp_col].values.astype('int64')
        chunk[timestamp_col] = pd.to_datetime(ns_vals * 1e6, unit='ns')

        chunk_start = chunk[timestamp_col].min()
        chunk_end = chunk[timestamp_col].max()

        if start_time is None:
            start_time = chunk_start
            end_time = chunk_end
        else:
            if chunk_start < start_time:
                start_time = chunk_start
            if chunk_end > end_time:
                end_time = chunk_end

        del chunk

    duration = end_time - start_time
    window_size = pd.Timedelta(minutes=window_minutes)
    total_windows = int(duration / window_size) + 1

    print(f"\n\n✅ Rango temporal identificado:")
    print(f"   Inicio: {start_time}")
    print(f"   Fin: {end_time}")
    print(f"   Duración: {duration}")
    print(f"   Ventanas ({window_minutes} min): {total_windows}")

    # ========================================================================
    # PASADA 2: Crear grafos con nodos FIJOS
    # ========================================================================

    print(f"\n📊 PASADA 2: Creando {total_windows} grafos temporales...")
    print(f"   Cada grafo tiene {num_nodes:,} nodos fijos\n")

    graphs_created = 0
    windows_skipped = 0

    graphs_file = f'{output_dir}/graphs_fixed_nodes.pkl'

    with open(graphs_file, 'wb') as f_out:

        for window_idx in range(total_windows):

            if window_idx % 10 == 0:
                percent = (window_idx / total_windows) * 100
                print(f"  Ventana {window_idx}/{total_windows} ({percent:.1f}%) | "
                      f"Grafos: {graphs_created} | Skip: {windows_skipped}")

            # Calcular rango de esta ventana
            window_start = start_time + window_idx * window_size
            window_end = window_start + window_size

            # Recolectar flows de esta ventana
            window_flows = []

            for chunk in read_filtered_chunks(pickle_file):
                # Corregir timestamps
                ns_vals = chunk[timestamp_col].values.astype('int64')
                chunk[timestamp_col] = pd.to_datetime(ns_vals * 1e6, unit='ns')

                # Filtrar por ventana
                mask = (
                    (chunk[timestamp_col] >= window_start) &
                    (chunk[timestamp_col] < window_end)
                )

                chunk_window = chunk[mask]

                if len(chunk_window) > 0:
                    window_flows.append(chunk_window)

                del chunk, chunk_window

            # Skip si no hay datos
            if len(window_flows) == 0:
                windows_skipped += 1
                continue

            window_df = pd.concat(window_flows, ignore_index=True)

            # Skip ventanas muy pequeñas
            if len(window_df) < 5:
                windows_skipped += 1
                del window_flows, window_df
                continue

            # ============================================================
            # CREAR GRAFO CON NODOS FIJOS
            # ============================================================

            graph = create_graph_fixed_nodes(
                window_df,
                src_ip_col,
                dst_ip_col,
                label_col,
                global_ip_to_idx,
                num_nodes
            )

            if graph is not None:
                # Añadir metadata temporal
                graph.window_id = window_idx
                graph.window_start = window_start
                graph.window_end = window_end

                # Guardar
                pickle.dump(graph, f_out)
                graphs_created += 1
            else:
                windows_skipped += 1

            # Liberar memoria
            del window_flows, window_df

            if window_idx % 20 == 0:
                gc.collect()

    print(f"\n\n{'='*70}")
    print("✅ PROCESAMIENTO COMPLETADO")
    print(f"{'='*70}")
    print(f"  Grafos creados: {graphs_created}")
    print(f"  Ventanas omitidas: {windows_skipped} (sin datos o muy pequeñas)")
    print(f"  Tasa de éxito: {(graphs_created/total_windows)*100:.1f}%")
    print(f"\n  Características de los grafos:")
    print(f"    Nodos por grafo: {num_nodes:,} (FIJO)")
    print(f"    Nodos activos: Variable según ventana")
    print(f"    Aristas: Variable según ventana")
    print(f"\n  Archivos generados:")
    print(f"    Grafos: {graphs_file}")
    print(f"    Tamaño: {os.path.getsize(graphs_file) / (1024**2):.2f} MB")
    print(f"    Mapeo IPs: {mapping_file}")

    # Metadata
    metadata = {
        'num_graphs': graphs_created,
        'num_nodes_per_graph': num_nodes,
        'window_minutes': window_minutes,
        'temporal_range': {
            'start': str(start_time),
            'end': str(end_time),
            'duration': str(duration)
        },
        'ip_to_idx_file': mapping_file,
        'graphs_file': graphs_file
    }

    # Guardar metadata
    metadata_file = f'{output_dir}/metadata.pkl'
    with open(metadata_file, 'wb') as f:
        pickle.dump(metadata, f)

    print(f"    Metadata: {metadata_file}")

    return graphs_created, metadata



# ============================================================================
# FUNCIÓN 2: Crear grafo individual con nodos fijos
# ============================================================================

def create_graph_fixed_nodes(window_df, src_col, dst_col, label_col,
                             global_ip_to_idx, num_total_nodes):
    """
    Crea un grafo con conjunto FIJO de nodos

    Args:
        window_df: DataFrame con flows de una ventana temporal
        src_col: Columna de IP origen
        dst_col: Columna de IP destino
        label_col: Columna de etiqueta
        global_ip_to_idx: Mapeo GLOBAL IP → índice (fijo)
        num_total_nodes: Número total de nodos (fijo)

    Returns:
        Data: Objeto PyTorch Geometric con grafo
    """

    # ========================================================================
    # 1. CONSTRUIR ARISTAS (solo flows presentes en ventana)
    # ========================================================================

    edge_list = []
    edge_features = []

    # Columnas de features para aristas (ajustar según tu CSV)
    edge_feat_cols = [
        'FLOW_DURATION_MILLISECONDS',
        'IN_BYTES',
        'IN_PKTS',
        'OUT_BYTES',
        'OUT_PKTS'
    ]

    # Filtrar solo columnas que existen
    edge_feat_cols = [c for c in edge_feat_cols if c in window_df.columns]

    if len(edge_feat_cols) == 0:
        # Si no hay features, usar al menos duración o defaults
        edge_feat_cols = ['FLOW_DURATION_MILLISECONDS'] if 'FLOW_DURATION_MILLISECONDS' in window_df.columns else []

    for _, row in window_df.iterrows():
        # Obtener índices GLOBALES de nodos
        src_idx = global_ip_to_idx[row[src_col]]
        dst_idx = global_ip_to_idx[row[dst_col]]

        edge_list.append([src_idx, dst_idx])

        # Features de la arista (del flow)
        edge_feat = []
        for col in edge_feat_cols:
            val = row[col]
            if pd.isna(val):
                val = 0.0
            edge_feat.append(float(val))

        # Si no hay features, usar dummy
        if len(edge_feat) == 0:
            edge_feat = [1.0]  # Dummy feature

        edge_features.append(edge_feat)

    # ========================================================================
    # 2. CONSTRUIR NODE FEATURES (para TODOS los nodos)
    # ========================================================================

    # Calcular actividad por cada IP en esta ventana
    node_activity = defaultdict(list)

    for _, row in window_df.iterrows():
        src_ip = row[src_col]
        dst_ip = row[dst_col]

        # Guardar info del flow para el nodo source
        node_activity[src_ip].append({
            'role': 'source',
            'bytes': row.get('IN_BYTES', 0),
            'pkts': row.get('IN_PKTS', 0),
            'duration': row.get('FLOW_DURATION_MILLISECONDS', 0)
        })

        # Guardar info para el nodo destination
        node_activity[dst_ip].append({
            'role': 'destination',
            'bytes': row.get('OUT_BYTES', 0),
            'pkts': row.get('OUT_PKTS', 0),
            'duration': row.get('FLOW_DURATION_MILLISECONDS', 0)
        })

    # Crear features para TODOS los nodos (incluyendo inactivos)
    node_features = []

    for ip in sorted(global_ip_to_idx.keys(), key=lambda x: global_ip_to_idx[x]):

        if ip in node_activity:
            # Nodo ACTIVO en esta ventana
            activities = node_activity[ip]

            # Agregaciones
            total_flows = len(activities)
            total_bytes = sum(a['bytes'] for a in activities)
            total_pkts = sum(a['pkts'] for a in activities)
            avg_duration = np.mean([a['duration'] for a in activities])

            # Separar por rol
            as_source = sum(1 for a in activities if a['role'] == 'source')
            as_dest = sum(1 for a in activities if a['role'] == 'destination')

            features = [
                float(total_flows),      # Total flows involucrados
                float(as_source),        # Flows como source
                float(as_dest),          # Flows como dest
                float(total_bytes),      # Total bytes
                float(total_pkts),       # Total packets
                float(avg_duration),     # Duración promedio
                1.0                      # Indicador de actividad
            ]
        else:
            # Nodo INACTIVO en esta ventana
            features = [
                0.0,  # total_flows
                0.0,  # as_source
                0.0,  # as_dest
                0.0,  # total_bytes
                0.0,  # total_pkts
                0.0,  # avg_duration
                0.0   # indicador de actividad
            ]

        node_features.append(features)

    # ========================================================================
    # 3. LABEL DEL GRAFO (clasificación de ventana)
    # ========================================================================

    # Detectar si hay ataque en esta ventana
    # Ajustar según tus labels específicos
    benign_labels = ['Benign', 'BENIGN', 'benign', 'Normal', 0]

    has_attack = ~window_df[label_col].isin(benign_labels).any()

    # Label binario
    y = torch.tensor([1 if has_attack else 0], dtype=torch.long)

    # ========================================================================
    # 4. CREAR OBJETO PyTorch Geometric Data
    # ========================================================================

    # Convertir a tensores
    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_features, dtype=torch.float)
    x = torch.tensor(node_features, dtype=torch.float)

    # Crear objeto Data
    data = Data(
        x=x,                        # [num_nodes, num_node_features]
        edge_index=edge_index,      # [2, num_edges]
        edge_attr=edge_attr,        # [num_edges, num_edge_features]
        y=y,                        # [1] - label del grafo
        num_nodes=num_total_nodes   # Número FIJO de nodos
    )

    # Metadata adicional
    data.num_active_nodes = len(node_activity)
    data.num_edges_actual = len(edge_list)

    return data


# ============================================================================
# FUNCIÓN 3: Leer grafos guardados (uno a la vez)
# ============================================================================

def read_graphs_generator(graphs_file):
    """
    Generador para leer grafos de uno en uno sin cargar todos en RAM
    """
    with open(graphs_file, 'rb') as f:
        while True:
            try:
                graph = pickle.load(f)
                yield graph
            except EOFError:
                break


# ============================================================================
# FUNCIÓN 4: Cargar todos los grafos (si caben en RAM)
# ============================================================================

def load_all_graphs(graphs_file):
    """
    Carga todos los grafos en una lista
    Solo usar si caben en RAM
    """
    graphs = []
    for graph in read_graphs_generator(graphs_file):
        graphs.append(graph)
    return graphs


# ============================================================================
# FUNCIÓN 5: Análisis de grafos creados
# ============================================================================

def analyze_graphs(graphs_file, sample_size=100):
    """
    Analiza características de los grafos sin cargar todos en RAM
    """

    print("\n" + "="*70)
    print("📊 ANÁLISIS DE GRAFOS CREADOS")
    print("="*70)

    stats = {
        'total_graphs': 0,
        'num_nodes': [],
        'num_active_nodes': [],
        'num_edges': [],
        'num_node_features': None,
        'num_edge_features': None,
        'labels': []
    }

    for i, graph in enumerate(read_graphs_generator(graphs_file)):

        stats['total_graphs'] += 1
        stats['num_nodes'].append(graph.num_nodes)
        stats['num_active_nodes'].append(graph.num_active_nodes)
        stats['num_edges'].append(graph.edge_index.shape[1])
        stats['labels'].append(graph.y.item())

        if stats['num_node_features'] is None:
            stats['num_node_features'] = graph.x.shape[1]
            stats['num_edge_features'] = graph.edge_attr.shape[1] if graph.edge_attr is not None else 0

        if i >= sample_size:
            break

    print(f"\n✅ Grafos analizados: {stats['total_graphs']}")

    print(f"\n📐 Dimensiones:")
    print(f"  Nodos por grafo: {stats['num_nodes'][0]:,} (FIJO)")
    print(f"  Node features: {stats['num_node_features']}")
    print(f"  Edge features: {stats['num_edge_features']}")

    print(f"\n📊 Nodos activos por ventana:")
    print(f"  Promedio: {np.mean(stats['num_active_nodes']):.1f}")
    print(f"  Mediana: {np.median(stats['num_active_nodes']):.1f}")
    print(f"  Min: {np.min(stats['num_active_nodes'])}")
    print(f"  Max: {np.max(stats['num_active_nodes'])}")

    print(f"\n🔗 Aristas por ventana:")
    print(f"  Promedio: {np.mean(stats['num_edges']):.1f}")
    print(f"  Mediana: {np.median(stats['num_edges']):.1f}")
    print(f"  Min: {np.min(stats['num_edges'])}")
    print(f"  Max: {np.max(stats['num_edges'])}")

    print(f"\n🏷️  Distribución de labels:")
    normal = stats['labels'].count(0)
    attack = stats['labels'].count(1)
    total = normal + attack

    print(f"  Normal (0): {normal:,} ({normal/total*100:.1f}%)")
    print(f"  Ataque (1): {attack:,} ({attack/total*100:.1f}%)")

    # Calcular densidad de grafos
    avg_edges = np.mean(stats['num_edges'])
    num_nodes = stats['num_nodes'][0]
    max_possible_edges = num_nodes * (num_nodes - 1)  # Grafo dirigido
    density = avg_edges / max_possible_edges * 100

    print(f"\n📈 Densidad del grafo:")
    print(f"  Aristas promedio: {avg_edges:.1f}")
    print(f"  Aristas máximas posibles: {max_possible_edges:,}")
    print(f"  Densidad: {density:.4f}% (sparse - ✅ BIEN)")

    return stats


# ============================================================================
# EJECUTAR PIPELINE COMPLETO
# ============================================================================

if __name__ == "__main__":

    import gc

    # Limpiar memoria
    gc.collect()

    print("\n🚀 INICIANDO CREACIÓN DE GRAFOS CON NODOS FIJOS\n")

    # Parámetros
    pickle_file = '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/filtered_chunks.pkl'
    output_dir = '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/graph_data_fixed_nodes'

    # Crear grafos
    num_graphs, metadata = create_graphs_with_fixed_nodes(
        pickle_file=pickle_file,
        timestamp_col=timestamp_col_start,
        src_ip_col=src_ip_col,
        dst_ip_col=dst_ip_col,
        label_col=label_col,
        window_minutes=10,
        output_dir=output_dir
    )

    # Analizar grafos creados
    if num_graphs > 0:
        graphs_file = metadata['graphs_file']
        stats = analyze_graphs(graphs_file)

        print(f"\n{'='*70}")
        print("🎉 PIPELINE COMPLETADO EXITOSAMENTE")
        print(f"{'='*70}")
        print(f"\n✅ {num_graphs} grafos listos para GNN+LSTM")
        print(f"\n📁 Archivos en: {output_dir}/")
        print(f"   - graphs_fixed_nodes.pkl")
        print(f"   - ip_to_idx_mapping.pkl")
        print(f"   - metadata.pkl")

    else:
        print("\n❌ No se crearon grafos - verifica los datos de entrada")